In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
print(pd.__version__)
print(np.__version__)
from imblearn.over_sampling import SMOTE
import re

2.3.3
2.3.5


In [2]:
cft_pdt_data = pd.read_csv("counterfeit_products_preprocessed_v2.csv")
cft_pdt_data.shape

(5000, 28)

In [3]:
indep_X = cft_pdt_data.drop('is_counterfeit', axis=1)
dep_Y = cft_pdt_data['is_counterfeit']

In [4]:
dep_Y.value_counts()

is_counterfeit
0    2952
1    2048
Name: count, dtype: int64

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
X_train.shape, X_test.shape

((3750, 27), (1250, 27))

In [6]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print("Train after SMOTE:", y_train.value_counts().to_dict())
print("Test (untouched):", y_test.value_counts().to_dict())

Train after SMOTE: {0: 2217, 1: 2217}
Test (untouched): {0: 735, 1: 515}


In [7]:
from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(score_func=chi2, k=10)
X_train_set = selector.fit_transform(X_train, y_train)
X_test_set = selector.transform(X_test)

selected_cols = indep_X.columns[selector.get_support()]
print(list(selected_cols))

['price', 'seller_reviews', 'product_images', 'description_length', 'shipping_time_days', 'domain_age_days', 'views', 'purchases', 'warranty_months', 'brand_suspicious']


In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train_set)
X_test_scaled = sc.transform(X_test_set)

In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
from sklearn.model_selection import GridSearchCV
param_grid = {'n_estimators': [50, 100, 200],'max_depth': [None, 5, 10, 20],'min_samples_split': [2, 5, 10],'criterion': ['gini', 'entropy']}

rf = RandomForestClassifier(random_state=0)
grid = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Params: {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Score: 0.7485360397616946


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:", confusion_matrix(y_test, y_pred))
print("\nClassification Report:", classification_report(y_test, y_pred))
print ("ROC-AUC:", roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1]))

Test Accuracy: 0.736

Confusion Matrix: [[600 135]
 [195 320]]

Classification Report:               precision    recall  f1-score   support

           0       0.75      0.82      0.78       735
           1       0.70      0.62      0.66       515

    accuracy                           0.74      1250
   macro avg       0.73      0.72      0.72      1250
weighted avg       0.73      0.74      0.73      1250

ROC-AUC: 0.7837051713889441


In [12]:
table=pd.DataFrame.from_dict(grid.cv_results_)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.923160,0.059043,0.035052,0.013680,gini,None,2,50,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.727170,0.733935,0.742954,0.750846,0.750564,0.741094,0.009310,12
1,1.805485,0.070645,0.041778,0.008407,gini,None,2,100,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.738444,0.740699,0.755355,0.750846,0.757336,0.748536,0.007649,1
2,4.117224,0.172292,0.127303,0.006211,gini,None,2,200,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.730552,0.748591,0.753100,0.755355,0.755079,0.748536,0.009312,2
3,1.022958,0.123780,0.021870,0.002154,gini,None,5,50,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.738444,0.723788,0.736189,0.750846,0.731377,0.736129,0.008909,23
4,2.123402,0.073761,0.071278,0.038115,gini,None,5,100,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.730552,0.729425,0.737317,0.747463,0.742664,0.737484,0.006920,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2.540263,0.318050,0.049742,0.022923,entropy,20,5,100,"{'criterion': 'entropy', 'max_depth': 20, 'min...",0.727170,0.738444,0.738444,0.739572,0.747178,0.738162,0.006392,17
68,6.335287,0.410143,0.110995,0.042542,entropy,20,5,200,"{'criterion': 'entropy', 'max_depth': 20, 'min...",0.730552,0.739572,0.746336,0.745209,0.742664,0.740866,0.005658,14
69,1.804313,0.422568,0.036824,0.005319,entropy,20,10,50,"{'criterion': 'entropy', 'max_depth': 20, 'min...",0.722661,0.723788,0.744081,0.731680,0.720090,0.728460,0.008717,36
70,3.266360,0.725039,0.050507,0.018529,entropy,20,10,100,"{'criterion': 'entropy', 'max_depth': 20, 'min...",0.727170,0.727170,0.745209,0.741826,0.733634,0.735002,0.007420,25


In [13]:
import pickle

pickle.dump(best_rf, open("counterfeit_model.sav", "wb"))
pickle.dump(sc, open("scaler.sav", "wb"))
pickle.dump(list(selected_cols), open("selected_features.sav", "wb"))

print("All 3 files saved separately!")

All 3 files saved separately!


In [14]:
def is_suspicious_brand(name):
    if not isinstance(name, str) or name.strip() == '':
        return 0
    if re.search(r'\d', name):
        return 1
    if re.search(r'[^A-Za-z0-9\s]', name):
        return 1
    return 0

In [16]:
def predict_counterfeit(price, seller_reviews, product_images, description_length,                                                                                                                                       
                         shipping_time_days, domain_age_days, views, purchases,                                                                                                                                            
                         warranty_months, brand_name):
    brand_suspicious = is_suspicious_brand(brand_name)
    user_input = pd.DataFrame([[price, seller_reviews, product_images, description_length,shipping_time_days, domain_age_days, views, purchases,warranty_months, brand_suspicious]],columns=selected_cols)                                                                                                                                                                                  

    user_input_scaled = sc.transform(user_input.values)                                                                                                                                                                                  
    prediction = best_rf.predict(user_input_scaled)                                                                                                                                                                                        
    prediction_proba = best_rf.predict_proba(user_input_scaled)                                                                                                                                                                              

    return {"prediction": "COUNTERFEIT" if prediction[0] == 1 else "GENUINE","counterfeit_probability": round(prediction_proba[0][1] * 100, 2),"brand_flagged_suspicious": bool(brand_suspicious)}